## 结论

本notebook离线读取固定官方公开task和已保存资料。参考版本0.1.1：78行、70 verified/8 provisional；70数值复算通过，4信用正例直接读取原始8-K，其余4无事件证明不完整。pass_flag表示名单、字段、保存字节及已实现公式一致，不代表完整首次发布证明或官方私有真值认证。信用数字真值保留null。


## 背景与方法

只使用Python标准库，不联网、不安装依赖。CPI和宏观修订按指定vintage；拍卖读取当日PDF；收益率同时回读Treasury XML与逐端点三个相邻ALFRED vintage。COT读取年度ZIP/API；EPS从SEC原始申报选单季GAAP diluted EPS。Yahoo close/adjclose属于同来源序列检查；IR展示服务另作美分价格对照，上游独立性仍未知。信用事件正文决定核查分支，候选标签不能跳过已保存的正例证据。补证清单research_checks保持研究记录范围。


In [1]:
from pathlib import Path
import hashlib, json, platform, subprocess, sys

PROJECT = Path.cwd().resolve()
while not (PROJECT / "tools/audit_reference_data.py").is_file():
    if PROJECT.parent == PROJECT:
        raise RuntimeError("请在 T4-WorkFlow 项目或其子目录运行 notebook")
    PROJECT = PROJECT.parent
DATA = PROJECT / "datasets/agenthon-t4-reference"
SCRIPT = PROJECT / "tools/audit_reference_data.py"
REFERENCE = DATA / "reference.jsonl"
REPORT = DATA / "qa/formula-audit.json"
print("Python:", sys.version)
print("Platform:", platform.platform())
print("项目:", PROJECT)
print("参考 SHA-256:", hashlib.sha256(REFERENCE.read_bytes()).hexdigest())

Python: 3.14.4 (main, Apr  7 2026, 13:13:20) [Clang 21.0.0 (clang-2100.0.123.102)]
Platform: macOS-26.5.2-arm64-arm-64bit-Mach-O
项目: .
参考 SHA-256: 7ab6d8ab80c9fe7f9b99d1ae023792b8f862eb7d48d84557e0e8c83eb1aa9621


## 资料

固定输入与 task 原始字节哈希位于 `inputs/source-manifest.json`；独立计算输入保存在 `sources/`。完整每行公式、原件定位、值、日期、版本分类和发现项保存在 `qa/formula-audit.json`。原 Company Facts 过滤文件中的 `source_sha256` 属完整上游响应；本审校不会把过滤副本误当该原始响应。EPS 另用完整 companyconcept raw 与对应元数据回读哈希。

In [2]:
command = [sys.executable, str(SCRIPT), "--dataset", str(DATA), "--reference", str(REFERENCE), "--output", str(REPORT), "--self-check", "--private-sources", str(PROJECT / ".local/private-evidence")]
print("精确执行参数:", json.dumps(command, ensure_ascii=False))
completed = subprocess.run(command, cwd=PROJECT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
completed.check_returncode()
audit = json.loads(REPORT.read_text(encoding="utf-8"))
assert audit["input"]["reference_sha256"] == hashlib.sha256(REFERENCE.read_bytes()).hexdigest()
assert audit["summary"]["pass_flag"]
assert audit["roster_validation"]["exact_roster_match"] and audit["roster_validation"]["unique_keys"]
assert not audit["hash_failures"]

精确执行参数: ["/opt/homebrew/opt/python@3.14/bin/python3.14", "./tools/audit_reference_data.py", "--dataset", "./datasets/agenthon-t4-reference", "--reference", "./datasets/agenthon-t4-reference/reference.jsonl", "--output", "./datasets/agenthon-t4-reference/qa/formula-audit.json", "--self-check", "--private-sources", "./.local/private-evidence"]
{
  "records": 78,
  "tasks": 11,
  "reference_status_counts": {
    "verified": 78,
    "provisional": 0,
    "unresolved": 0
  },
  "row_integrity_pass": 78,
  "independent_numeric_calculation_pass": 70,
  "independent_numeric_calculation_fail": 0,
  "no_independent_numeric_calculation": 8,
  "independent_positive_credit_events_confirmed": 4,
  "designated_vintage_or_release_archive_supported": 49,
  "provenance_class_counts": {
    "bounded_public_negative_case": 4,
    "current_historical_series_finite_window_verified": 3,
    "current_historical_series_unknown_vintage": 10,
    "dated_original_event_document": 4,
    "dated_original_release_do

## 结果

下面输出来自本次执行的审校结果；每行实际数值读取原件后独立计算，未直接复制 `reference.inputs`。无独立数字计算的信用标签明确显示为空。

In [3]:
print(json.dumps(audit["summary"], ensure_ascii=False, indent=2))
print("\n按 task 统计：")
for task_id in sorted({r["task_id"] for r in audit["records"]}):
    rows = [r for r in audit["records"] if r["task_id"] == task_id]
    print(task_id, "rows=", len(rows), "数值复算通过=", sum(r["calculation_pass"] is True for r in rows), "行检查通过=", sum(r["pass_flag"] for r in rows))
print("\n内存错误注入检查：")
print(json.dumps(audit["negative_controls"], ensure_ascii=False, indent=2))

{
  "records": 78,
  "tasks": 11,
  "reference_status_counts": {
    "verified": 78,
    "provisional": 0,
    "unresolved": 0
  },
  "row_integrity_pass": 78,
  "independent_numeric_calculation_pass": 70,
  "independent_numeric_calculation_fail": 0,
  "no_independent_numeric_calculation": 8,
  "independent_positive_credit_events_confirmed": 4,
  "designated_vintage_or_release_archive_supported": 49,
  "provenance_class_counts": {
    "bounded_public_negative_case": 4,
    "current_historical_series_finite_window_verified": 3,
    "current_historical_series_unknown_vintage": 10,
    "dated_original_event_document": 4,
    "dated_original_release_document": 7,
    "designated_historical_vintage": 35,
    "earliest_filing_xbrl_observation": 15
  },
  "hash_files_checked_against_prior_declaration": 441,
  "hash_failures": 0,
  "pass_flag": true,
  "supplement_manifest_count": 7,
  "supplement_validation_errors": 0,
  "pass_flag_scope": "roster/字段/源哈希/已实现数值公式一致；不表示所有78行已证明首次发布或完整无事件",
  "f

In [4]:
print("task_id | entity_id | reference | 独立计算 | label | 复算pass | 行pass | 版本证据类型")
for row in audit["records"]:
    def number(value):
        return "null" if value is None else format(value, ".12g")
    print(" | ".join([row["task_id"], row["entity_id"], number(row["reference_value"]), number(row.get("computed_value")), str(row["reference_label"]), str(row["calculation_pass"]), str(row["pass_flag"]), row["provenance_class"]]))

task_id | entity_id | reference | 独立计算 | label | 复算pass | 行pass | 版本证据类型
t4-EXAMPLE-eps-beat | AAPL | 1.53 | 1.53 | inline | True | True | earliest_filing_xbrl_observation
t4-auction-btc-202411-us7 | AUC_10Y_20241105 | 2.58 | 2.58 | None | True | True | dated_original_release_document
t4-auction-btc-202411-us7 | AUC_20Y_20241120 | 2.34 | 2.34 | None | True | True | dated_original_release_document
t4-auction-btc-202411-us7 | AUC_2Y_20241125 | 2.77 | 2.77 | None | True | True | dated_original_release_document
t4-auction-btc-202411-us7 | AUC_30Y_20241106 | 2.64 | 2.64 | None | True | True | dated_original_release_document
t4-auction-btc-202411-us7 | AUC_3Y_20241104 | 2.6 | 2.6 | None | True | True | dated_original_release_document
t4-auction-btc-202411-us7 | AUC_5Y_20241126 | 2.43 | 2.43 | None | True | True | dated_original_release_document
t4-auction-btc-202411-us7 | AUC_7Y_20241127 | 2.71 | 2.71 | None | True | True | dated_original_release_document
t4-cotpos-202411-us10 | CORN_CBT | 7

In [5]:
print("\n公开资料与首次发布边界：")
for finding in audit["material_findings"]:
    print("-", finding)
print("\nAMGN 日期语义：")
amgn = next(r for r in audit["records"] if r["task_id"] == "t4-eps-yoy-2023Q2-mixed" and r["entity_id"] == "AMGN")
print(json.dumps(amgn["date_semantics"], ensure_ascii=False, indent=2))
print("状态:", amgn["reference_status"], "季度值:", amgn["computed_value"], "标签:", amgn["computed_label"])
print("\n信用无事件证明：")
for row in audit["records"]:
    if row["provenance_class"] == "negative_proof_incomplete":
        print(row["entity_id"], row["reference_status"], "数字真值=", row["reference_value"], "proof_complete=", row["proof_complete"])
print("\n具下载声明但原正文未保留：")
for source in audit["declared_but_unpreserved_sources"]:
    print(source["status"], source.get("path"), source.get("url"))
print("\n补证保存字节与研究结论边界：")
for supplement in audit["supplementary_evidence"]:
    print(supplement["manifest"], "保存来源数=", supplement["preserved_source_count"], supplement["scope"])



公开资料与首次发布边界：
- verified 是公开来源研究状态，不能自动视为 first-published provenance proof。
- CPI 使用发布日 vintage 指数；BLS 首稿显示0.1个百分点精度，派生值不等于官方内部未舍入值。
- 宏观同月不同 resolving vintage 逐行保留；macrorev Development 官方 numeric truth 缺失，私有评分未比对。
- COT年度/API、Treasury曲线当前历史 XML 和 Yahoo 当前历史快照未提供完整首次发布修订链。
- EPS SEC 原始申报季度观察值与题面 prior 分开；过滤快照的 source_sha256 不是过滤副本字节哈希。
- 收益率补证直接读取三个相邻历史vintage；ALFRED可见日不等于Treasury全球首发时点。
- 补证清单内research_checks保持研究记录范围；脱敏副本hash只核对保存字节，不回读已删除的原始HTTP响应。
- AMGN 2023Q2 实际8/3公告晚于题面8/2；官方#27已澄清resolution/expected report为context，指定季度结果对齐且可默认纳入。首发版本与私有outcome仍未确认。
- 信用负例按有限公开核查范围验收；无事件非穷尽保证，不能把本次 pass_flag 解读为 proof complete。
- Yahoo 日线开盘 timestamp 只定位交易日，历史 NY 时区独立换算；无 vendor events 不等于完整公司行动证明。

AMGN 日期语义：
{
  "cutoff_date": "2023-07-14",
  "resolution_date": "2023-08-02",
  "quarter_start": "2023-04-01",
  "quarter_end": "2023-06-30",
  "first_selected_SEC_filed_date": "2023-08-04",
  "declared_release_dates": [
    "2023-08-03"
  ],
  "declared_schedule_notice_dates": [
    "2023-07-31"
  ]

## 证据边界

35行有明确ALFRED历史vintage、7个拍卖比率有当日原PDF、4个信用正例有原始事件正文、15行EPS有最早SEC季度观察值。13行COT/历史行情仍缺完整首发链，4个信用负例尚不能证明全窗无事件。ALFRED最早测试可见日不等于Treasury全球首发时间；保留global_source_first_publication_proven=false。AMGN 8月3日公告晚于题面8月2日resolution，保持暂定。

全部78核心数值/标签/单位/质量状态与上一版本相同；更新来源与口径说明导致参考集hash变化。fit_ready_without_additional_provenance=false。未获取私有outcome或执行官方平台计分。
